# Assistant
Looking at the selected code:


In [ ]:
pip install ipykernel


This is a pip installation command for the `ipykernel` package, which is typically used in Jupyter notebooks to provide the Python kernel. Since this is a command rather than Python code to refactor, I'll provide some improvements for how this could be written in a Python script:


In [ ]:
"""
Install the ipykernel package required for Jupyter notebook Python kernel.

This script demonstrates how to programmatically install Python packages.
"""
import sys
import subprocess
from typing import List, Optional


def install_package(package_name: str, upgrade: bool = False, quiet: bool = False) -> Optional[str]:
    """
    Install a Python package using pip.
    
    Args:
        package_name: Name of the package to install
        upgrade: Whether to upgrade the package if already installed
        quiet: Whether to suppress output
        
    Returns:
        Output from the pip install command or None if an error occurred
    """
    cmd: List[str] = [sys.executable, "-m", "pip", "install"]
    
    if upgrade:
        cmd.append("--upgrade")
    
    if quiet:
        cmd.append("--quiet")
        
    cmd.append(package_name)
    
    try:
        result = subprocess.check_output(cmd, stderr=subprocess.STDOUT)
        return result.decode('utf-8')
    except subprocess.CalledProcessError as e:
        print(f"Error installing {package_name}: {e}")
        return None


# Example usage
if __name__ == "__main__":
    result = install_package("ipykernel")
    if result:
        print(f"Successfully installed ipykernel")


This refactored version:
1. Provides a reusable function for package installation
2. Includes proper docstrings and type hints
3. Handles errors gracefully
4. Offers options for upgrading packages and controlling output verbosity

Note: If you're working in a Jupyter notebook, you'd typically use the `!pip install ipykernel` magic command directly in a cell rather than this Python function approach.

In [11]:
import os
import zipfile

zip_path = r'C:\Users\Asus\OneDrive\Desktop\validation_frames.zip'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall() 
        print("✅ Extraction complete.")
else:
    print(f"File not found: {zip_path}")



✅ Extraction complete.


In [ ]:
!pip3 install tensorflow tensorflow-gpu numpy matplotlib pillow opencv-python


In [ ]:
!pip install scikit-learn


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

In [13]:
import os
import cv2
import numpy as np


dataset_dir = r'C:\Users\Asus\OneDrive\Desktop\archive\Real Life Violence Dataset'
violence_dir = os.path.join(dataset_dir, 'Violence')
nonviolence_dir = os.path.join(dataset_dir, 'NonViolence')


def extract_frames(video_path, frames_dir, frame_rate=5):
    video_name = os.path.basename(video_path).split('.')[0]
    frame_dir = os.path.join(frames_dir, video_name)
    os.makedirs(frame_dir, exist_ok=True)
    
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % frame_rate == 0:  # Capture every nth frame
            cv2.imwrite(os.path.join(frame_dir, f"{frame_count}.jpg"), frame)
        frame_count += 1
    cap.release()


frames_dir = 'frames'
os.makedirs(frames_dir, exist_ok=True)

violence_frames_dir = os.path.join(frames_dir, 'Violence')
nonviolence_frames_dir = os.path.join(frames_dir, 'NonViolence')
os.makedirs(violence_frames_dir, exist_ok=True)
os.makedirs(nonviolence_frames_dir, exist_ok=True)

for video in os.listdir(violence_dir):
    extract_frames(os.path.join(violence_dir, video), violence_frames_dir)

for video in os.listdir(nonviolence_dir):
    extract_frames(os.path.join(nonviolence_dir, video), nonviolence_frames_dir)


In [ ]:
import os
import shutil
import random


frames_dir = r'C:\Users\Asus\OneDrive\Desktop\frames'
validation_dir = 'validation_frames001'


subdirs = ['Violence', 'NonViolence']


for subdir in subdirs:
    os.makedirs(os.path.join(validation_dir, subdir), exist_ok=True)


for subdir in subdirs:
    frames_path = os.path.join(frames_dir, subdir)
    validation_path = os.path.join(validation_dir, subdir)
    

    files = os.listdir(frames_path)
    

    validation_files = random.sample(files, int(len(files) * 0.2))
    

    for file in validation_files:
        shutil.move(os.path.join(frames_path, file), validation_path)


In [15]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Sequential
from sklearn.metrics import accuracy_score, classification_report


train_dir = r'C:\Users\Asus\OneDrive\Desktop\frames'
validation_dir = r'C:\Users\Asus\OneDrive\Desktop\validation_frames001'  


train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

validation_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='binary'
)

validation_generator = validation_datagen.flow_from_directory(
        validation_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='binary'
)


model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


history = model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // 32,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // 32,
    epochs=10
)


model.save('violence_detection_model002.h5')


Found 30649 images belonging to 2 classes.
Found 6736 images belonging to 2 classes.
Epoch 1/10
957/957 [==============================] - 1368s 1s/step - loss: 0.3759 - accuracy: 0.8198 - val_loss: 0.3724 - val_accuracy: 0.8449
Epoch 2/10
957/957 [==============================] - 993s 1s/step - loss: 0.1774 - accuracy: 0.9294 - val_loss: 0.3487 - val_accuracy: 0.8735
Epoch 3/10
957/957 [==============================] - 1117s 1s/step - loss: 0.1147 - accuracy: 0.9566 - val_loss: 0.3405 - val_accuracy: 0.8854
Epoch 4/10
957/957 [==============================] - 1327s 1s/step - loss: 0.0965 - accuracy: 0.9640 - val_loss: 0.3940 - val_accuracy: 0.8868
Epoch 5/10
957/957 [==============================] - 1396s 1s/step - loss: 0.0708 - accuracy: 0.9745 - val_loss: 0.4931 - val_accuracy: 0.8720
Epoch 6/10
957/957 [==============================] - 1434s 1s/step - loss: 0.0645 - accuracy: 0.9781 - val_loss: 0.5153 - val_accuracy: 0.8729
Epoch 7/10
957/957 [==============================] 

In [ ]:
!pip install --upgrade pip

In [ ]:
!pip3 install --upgrade pip

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model


model = load_model('violence_detection_model.h5')


cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break
    

    frame_small = cv2.resize(frame, (224, 224))
    frame_small = frame_small / 255.0
    

    frame_expanded = np.expand_dims(frame_small, axis=0)
    

    predictions = model.predict(frame_expanded)
    prediction = "Violence" if predictions[0][0] > 0.5 else "Non-Violence"
    confidence = predictions[0][0] if predictions[0][0] > 0.5 else 1 - predictions[0][0]
    confidence = round(confidence * 100, 2)
    

    cv2.putText(frame, f"{prediction} - {confidence}%", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0) if prediction == "Non-Violence" else (0, 0, 255), 2)
    

    cv2.imshow('Webcam Feed', frame)
    

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()
